# Capstone Phase 3 牛津 Tutorial LLM 仿真 (v6.0)

> **Oxford Tutorial + HBS Devil's Advocate + Hattie 4-Level Formative Feedback**

## Persona Prompt (System Role for the Socratic Tutor)

You are an **Oxford tutorial fellow** in **Capstone Phase 3: Agentic system architecture, LangGraph orchestration, multi-agent design, human-in-loop**. You conduct one-on-one tutorials in the Socratic tradition.

**Rules you MUST follow every turn**:
1. **Never give direct answers.** 你不直接给答案, 不写完整代码, 不替学生下结论。
2. **Use Socratic questioning.** 每轮至少一个追问: 为什么 / 反例 / 若前提变 / 凭什么 / 如何 / 假设...变 / 依据。
3. **Act as HBS devil's advocate.** 主动质疑学生的隐含假设, 反问"如果反向成立呢?"。
4. **Reject vague claims.** 学生说"差不多""应该可以""大概"时, 拒绝接受, 要求给出具体 LangGraph API 名/字段名/节点名。
5. **End each turn with a probing question.** 每轮末尾抛一个开放性追问。
6. **限频**: 每单元每天 1 次正式 tutorial, 防依赖 (见 cell6)。

本 tutorial 不调任何真实 LLM API。Socratic 追问用**静态 if/else 分支模拟** (见 cell3), 聚焦本单元领域概念: LangGraph StateGraph / add_conditional_edges / interrupt_before / MemorySaver / MCP / A2A / Plan-Execute / 天道推演 × 多Agent仿真。


## cell2: Pre-Tutorial Task (强制 Retrieval Practice)

> 牛津 tutorial 的核心: 学生必须先提交一段 essay/解题/方案, tutor 才能"挑战"它。本单元的 pre-tutorial task 是**强制提取练习** (retrieval practice, 优于重读)。

**提交要求** (在下方代码块写好后运行, 写入 `student_model.json` 的 `pre_tutorial_essay` 字段):

1. **300 字 essay**: 用 Capstone 三层架构 (用户交互/Agent编排/数据知识) 解释 `researcher -> strategist -> writer -> review -> publish` 五节点分别属于哪层, 以及 `knowledge_context` 字段如何从 Phase 2 数据知识层桥接到 Phase 3 Agent编排层。
2. **一段代码片段** (伪代码或真实 LangGraph): 写出 `route_after_review(state)` 路由函数, 含三个分支: `safety_flag=True -> publish` / `revision_count >= 3 -> publish` / 否则 -> `writer`。
3. **一句话反思**: 你觉得 HITL `interrupt_before=["review_node"]` 的 Capstone 治理意义是什么? (不许说"差不多", 必须具体)

写完才可进入 cell3 Socratic 追问环节。Tutor 会基于你的 essay 内容定向追问。


In [ ]:
# cell3: Multi-turn Socratic Loop (>=4 rounds, STATIC if/else, NO real API call)
# 模拟牛津 tutor: 不直接给答案, 用 >=5 个苏格拉底问 追问学生 essay 中的盲点

import json, os
from pathlib import Path

STUDENT_ESSAY = """
[在此粘贴你的 pre-tutorial essay - 三层架构映射 + route_after_review 代码 + HITL 反思]
"""

# 苏格拉底追问库 (>=5 个: 为什么/反例/若前提变/凭什么/如何)
SOCRATIC_QUESTIONS = [
    "Q1 (为什么): 你的 essay 说 writer 在 Agent编排层, 凭什么不在数据知识层? writer 读取了哪个字段? 这个字段产自哪层?",
    "Q2 (反例): 若把 `route_after_review` 的退出条件 `revision_count >= 3` 改成 `>= 100`, 系统会发生什么? 这反过来证明了什么?",
    "Q3 (若前提变): 若 LangGraph 没有 `MemorySaver` checkpointer, 你的三步 HITL (invoke -> update_state -> resume) 哪一步会失败? 失败的具体表现是什么?",
    "Q4 (凭什么): 你说 MCP 解决 researcher 工具调用, A2A 解决 Agent 通信。凭什么两者是互补而不是替代? 给一个同时需要两者的场景。",
    "Q5 (如何): 如何把'天道推演 × 多Agent仿真'的'记录假设/追踪偏差/更新因果模型'三步映射到 LangGraph 的 Checkpointing + 条件边 + 反馈学习节点? 每步对应哪个 API?",
    "Q6 (依据): 你的 route_after_review 返回值是 True/False 还是 'publish'/'writer'? add_conditional_edges 的字典键必须是什么类型? 依据 LangGraph 哪条规则?"
]

# 静态模拟 Socratic 追问 - 不调 openai/anthropic API
def socratic_round(round_idx, student_answer):
    """根据学生当前答案, 模拟 tutor 的 Socratic 追问。
    用 if/else 分支判断学生答案中的关键词, 给出领域特定的反问。"""
    ans = (student_answer or "").lower()
    if round_idx == 0:
        # Round 1: 总是从 Q1 开始 - 探测 essay 是否真分层
        return SOCRATIC_QUESTIONS[0]
    elif round_idx == 1:
        # Round 2: 根据学生对 Q1 的回答分支
        if "knowledge_context" in ans and "phase 2" in ans:
            return SOCRATIC_QUESTIONS[1]  # 已答对 -> 升级到反例
        else:
            return "你的回答没提到 knowledge_context 字段。请重新回答 Q1: writer 读取的具体字段名是什么? 该字段从哪一层流过来?"
    elif round_idx == 2:
        # Round 3: 探测 Checkpointing 理解
        if "checkpoint" in ans or "memorysaver" in ans:
            return SOCRATIC_QUESTIONS[2]  # 已有概念 -> 问反例
        else:
            return SOCRATIC_QUESTIONS[3]  # 概念缺失 -> 转 MCP/A2A
    elif round_idx == 3:
        # Round 4: 探测天道推演映射
        if "mcp" in ans and "a2a" in ans:
            return SOCRATIC_QUESTIONS[4]  # 已懂 MCP/A2A -> 升级天道推演
        else:
            return "你的回答混淆了 MCP 和 A2A。请明确: MCP 接的是工具还是 Agent? A2A 接的是工具还是 Agent? 为什么不能互换?"
    else:
        # Round 5+: 收尾问 - 探测 API 细节
        return SOCRATIC_QUESTIONS[5]

# 模拟 4 轮 Socratic 对话 (学生可在此填自己的答案)
student_answers = [
    "writer 在 Agent编排层, 读取 knowledge_context 字段, 该字段从 Phase 2 数据知识层流过来",
    "若 revision_count >= 100, writer 会无限循环修改文案, 直到达到 LangGraph 默认递归上限报错 (默认 25)",
    "MemorySaver 持久化 State, 没有它 update_state 找不到 thread 状态, 第二步失败",
    "MCP 接工具 (researcher 调知识图谱), A2A 接 Agent (跨进程 Agent 通信), 两者互补: LangGraph 进程内用 State 共享 (同进程 A2A), 跨进程用 A2A 协议"
]

print("=" * 70)
print("Oxford Tutorial Simulation - Capstone Phase 3 (STATIC, no API call)")
print("=" * 70)
for i in range(4):
    tutor_q = socratic_round(i, student_answers[i] if i < len(student_answers) else "")
    print(f"\n[Round {i+1}]")
    print(f"  Student: {student_answers[i] if i < len(student_answers) else '(待答)'}")
    print(f"  Tutor (Socratic): {tutor_q}")

print("\n" + "=" * 70)
print("Tutorial 结束 - 进入 cell4 写入 student_model.json")


In [ ]:
# cell4: student_model.json Read/Write (记录掌握度/盲点)
# 字段: mastery (0-1), blind_spots (list), drill_scores (dict), last_tutorial_date

import json, os
from datetime import date

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "capstone-phase-3-agentic-system-architecture",
        "mastery": 0.0,
        "blind_spots": [],
        "drill_scores": {"A1": None, "B1": None, "C1": None},
        "ilo_scores": {"ILO-1": None, "ILO-2": None, "ILO-3": None, "ILO-4": None, "ILO-5": None},
        "pre_tutorial_essay": None,
        "last_tutorial_date": None,
        "tutorial_count_today": 0
    }

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

# 模拟一次 tutorial 后的更新 (基于 cell3 的 4 轮 Socratic 答对情况)
model = load_student_model()

# 假设 cell3 学生答对 3/4 轮 (Q1/Q2/Q3 对, Q4 MCP/A2A 混淆)
model["mastery"] = 0.75  # 3/4 = 0.75, 未达 0.80 mastery
model["blind_spots"] = [
    {"drill_id": "B1", "fail_count": 1, "blind_spot": "MCP vs A2A 边界混淆 - MCP接工具, A2A接Agent, 不可互换"},
    {"drill_id": "C1", "fail_count": 0, "blind_spot": "天道推演映射尚未在 Socratic 中验证 (Round 5 未触发)"}
]
model["drill_scores"] = {"A1": 0.85, "B1": 0.60, "C1": 0.75}  # B1 < 0.60 触发 weak_loop
model["ilo_scores"] = {"ILO-1": 0.85, "ILO-2": 0.60, "ILO-3": 0.75, "ILO-4": 0.50, "ILO-5": None}
model["pre_tutorial_essay"] = "(学生已提交, 见 cell2)"
model["last_tutorial_date"] = str(date.today())
model["tutorial_count_today"] = 1

save_student_model(model)
print(f"student_model.json 已写入:")
print(f"  mastery: {model['mastery']} (mastery 阈值 0.80, {'PASS' if model['mastery'] >= 0.80 else 'FAIL - 需补做'})")
print(f"  blind_spots: {len(model['blind_spots'])} 个")
print(f"  drill B1: {model['drill_scores']['B1']} (< 0.60 -> 触发 weak_loop, 见 practice.md)")
print(f"  ILO-4: {model['ilo_scores']['ILO-4']} (< 0.70 -> 该 ILO 未掌握)")


## cell5: Hattie 4-Level Formative Feedback (避免 Self 级表扬)

> Hattie (2009) 反馈模型: Task / Process / Self-Reg / Self。Self 级 (表扬"你真聪明") 对学习无效甚至有害, 本 tutorial 跳过 Self 级, 只用前 3 级 + Feed-Forward。

基于 cell3 的 4 轮 Socratic 表现 + cell4 student_model.json, 给出四级反馈:

In [ ]:
# cell5: Hattie 4-Level Feedback (基于 cell3/cell4 的具体表现, 不用 Self 级表扬)

model = json.load(open("student_model.json", "r", encoding="utf-8"))

print("[TASK] 任务级反馈 - 针对具体任务错误:")
print("  - route_after_review 代码: 你写的返回值是 True/False, 但 add_conditional_edges 的字典键")
print("    必须与返回值匹配。LangGraph 推荐返回字符串 'publish'/'writer'/'exit', 字典键用同名字符串。")
print("  - 修正: route_after_review 应返回 'publish' if safety_flag else ('exit' if revision_count>=3 else 'writer')")
print()
print("[PROCESS] 过程级反馈 - 针对学习策略/方法:")
print("  - 你在 Q1-Q3 表现良好 (三层架构 + 循环退出 + Checkpointing 概念清晰),")
print("    但 Q4 (MCP vs A2A) 混淆了'接工具'和'接 Agent'。这说明你的对比学习策略不够:")
print("    建议用 Venn 图对比 MCP (工具层) vs A2A (Agent层) vs LangGraph State (进程内 Agent 通信)。")
print("  - ILO-4 (Plan-Execute/MCP/A2A) 是你的弱项, 触发 schedule.json C2/C3 卡片额外复习。")
print()
print("[SELF-REG] 自我调节反馈 - 针对元认知/监控:")
print("  - 你在 cell3 第 4 轮回答时, 没有先自问'我对 MCP/A2A 概念清楚吗'就直接作答, 导致混淆。")
print("  - 修正: 进入下一轮 Socratic 前, 先在脑中过一遍'我确定的是什么, 不确定的是什么',")
print("    对不确定的部分主动反问 tutor, 而非被动等待 tutor 追问。")
print("  - 元认知监控 (monitoring) 是 mastery 学习的核心, 比 task 知识本身更可迁移。")
print()
print("[FEED-FORWARD] 前馈反馈 - 针对下一步行动:")
print("  - 立即行动: 回到 practice.md drill_B1 Faded 阶段重做, 重点练习 route_after_review 返回值类型")
print("  - 24h 内: 复习 schedule.json C2 (MCP/A2A) + C3 (Plan-Execute) 卡片, 用 SM-2 算法重排")
print("  - 48h 内: 重做 cell3 Socratic 第 4 轮, 验证 MCP/A2A 区分已修正")
print("  - 若下次 tutorial 仍混淆 MCP/A2A -> 触发 weak_loop, 回退到 Worked Example (notes.md 前沿节重读)")
print()
print("(无 [SELF] 表扬级 - Hattie 研究表明 '你真聪明' 类表扬降低后续挑战意愿)")


## cell6: 限频 + Exit Artifact

### 限频 (防依赖)

- 每单元每天**最多 1 次**正式 tutorial (cell3 Socratic 4 轮 + cell5 Hattie 反馈)
- 防依赖: Socratic 是脚手架, 不是答案机。频繁依赖会削弱独立 retrieval 能力。
- 当日已用 1 次: `student_model.json` 的 `tutorial_count_today` 字段会自增, 超过 1 次本脚本拒绝再次启动 Socratic loop, 提示"明日再来, 今日先做 practice.md drill 或 schedule.json 卡片复习"。
- 次日 0 点 `tutorial_count_today` 重置为 0。

### Exit Artifact (强制输出, 防"听完就忘")

完成本 tutorial 后, 在 `student_model.json` 的 `exit_artifact` 字段写入:

1. **2-3 个盲点** (基于 cell3 Socratic 答错 + cell5 [TASK]/[PROCESS] 反馈): 例 ["MCP vs A2A 边界", "route_after_review 返回值类型", "天道推演映射第3步"]
2. **推荐复习单元**: 基于盲点定位前置单元
   - 若盲点是 "TypedDict/Annotated" -> 回 Phase 2 数据表示单元
   - 若盲点是 "条件边/路由" -> 回技能5 Day2 LangGraph 编排单元
   - 若盲点是 "HITL 治理意义" -> 回技能2 Day2 企业编排单元
   - 若盲点是 "MCP/A2A" -> 重读本单元 notes.md「2026前沿」节 + schedule.json C2 卡片
3. **下次 tutorial 的预热问题** (1 个): 学生自己抛一个还想深挖的问题, 例"如果加一个'消费者 Agent'节点, 状态图怎么改?"

### 验收

- `student_model.json` 含 `exit_artifact` 字段且盲点 >= 2 个 -> 本单元 tutorial 收敛
- 盲点为 0 个 -> tutor 拒绝结束, 反问"你确定没有任何盲点? 那请回答: 如果 LangGraph 默认递归上限改成 5, 你的 revision_count>=3 退出条件还合理吗?" (总有盲点, mastery 是渐进的)

---

*本 notebook 由 v6.0 学习科学层升级生成。Socratic loop 为静态 if/else 模拟, 不调任何真实 LLM API。所有追问领域特定 (LangGraph/MCP/A2A/Plan-Execute/天道推演 × 多Agent仿真), 不使用通用模板。*
